# Práctica Final NLP — Preprocesado de texto

## Instalaciones e imports

In [2]:
!pip install gensim --quiet

import os
import re
import string
import unicodedata
import numpy as np
import pandas as pd
import urllib.request
import nltk

nltk.download('stopwords')
nltk.download('punkt')

from nltk.corpus import stopwords
from nltk.stem.snowball import EnglishStemmer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 67.7 MB/s eta 0:00:00


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


## Carga del corpus

In [3]:
files = [
    "reviews_Clothing_Shoes_and_Jewelry_5.json.gz",
    "reviews_Office_Products_5.json.gz",
    "reviews_Video_Games_5.json.gz"
]
base_url = "http://snap.stanford.edu/data/amazon/productGraph/categoryFiles/"

for f in files:
    if not os.path.exists(f):
        print(f"Descargando {f}...")
        urllib.request.urlretrieve(base_url + f, f)

df_clothing   = pd.read_json("reviews_Clothing_Shoes_and_Jewelry_5.json.gz", lines=True, compression='gzip')
df_office     = pd.read_json("reviews_Office_Products_5.json.gz", lines=True, compression='gzip')
df_videogames = pd.read_json("reviews_Video_Games_5.json.gz", lines=True, compression='gzip')

for df_ in [df_clothing, df_office, df_videogames]:
    df_['sentiment'] = df_['overall'].apply(lambda x: 0 if x < 3 else 1)

N = 10000
df = pd.concat([
    df_clothing[['reviewText','overall','sentiment']].dropna().sample(N, random_state=42),
    df_office[['reviewText','overall','sentiment']].dropna().sample(N, random_state=42),
    df_videogames[['reviewText','overall','sentiment']].dropna().sample(N, random_state=42)
]).reset_index(drop=True)

print("Corpus cargado:", df.shape)

Descargando reviews_Clothing_Shoes_and_Jewelry_5.json.gz...
Descargando reviews_Office_Products_5.json.gz...
Descargando reviews_Video_Games_5.json.gz...
Corpus cargado: (30000, 3)


## Funciones auxiliares de preprocesado

Se definen funciones específicas para cada tarea de normalización:
- `to_lowercase`: convierte a minúsculas
- `remove_accents`: elimina acentos y caracteres especiales
- `remove_punctuation`: elimina signos de puntuación
- `remove_numbers`: elimina tokens numéricos
- `remove_stopwords`: filtra palabras vacías (stopwords en inglés)
- `apply_stemming`: reduce palabras a su raíz morfológica
- `remove_short_words`: elimina tokens de longitud < 2

In [4]:
sw = set(stopwords.words('english'))
stemmer = EnglishStemmer()

def to_lowercase(text):
    """Convierte el texto a minúsculas."""
    return text.lower()

def remove_accents(text):
    """Elimina acentos y caracteres especiales."""
    return unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8', 'ignore')

def remove_punctuation(text):
    """Elimina signos de puntuación."""
    table = str.maketrans('', '', string.punctuation)
    return text.translate(table)

def remove_numbers(text):
    """Elimina tokens numéricos."""
    return re.sub(r'\b\d+\b', '', text)

def remove_stopwords(tokens):
    """Elimina stopwords de una lista de tokens."""
    return [w for w in tokens if w not in sw]

def apply_stemming(tokens):
    """Aplica stemming a una lista de tokens."""
    return [stemmer.stem(w) for w in tokens]

def remove_short_words(tokens, min_len=2):
    """Elimina tokens demasiado cortos."""
    return [w for w in tokens if len(w) >= min_len]

## Pipeline de preprocesado

La función `preprocess_text` integra todas las funciones anteriores
en un pipeline secuencial. El parámetro `stem=False` permite activar
o desactivar el stemming según las necesidades del modelo posterior.

In [5]:
def preprocess_text(text, stem=False):
    """
    Pipeline completo de preprocesado de texto:
    1. Minúsculas
    2. Eliminar acentos
    3. Eliminar puntuación
    4. Eliminar números
    5. Tokenización
    6. Eliminar stopwords
    7. Eliminar palabras cortas
    8. Stemming
    """
    text = to_lowercase(text)
    text = remove_accents(text)
    text = remove_punctuation(text)
    text = remove_numbers(text)
    tokens = text.split()
    tokens = remove_stopwords(tokens)
    tokens = remove_short_words(tokens)
    if stem:
        tokens = apply_stemming(tokens)

    return ' '.join(tokens)

## Ejemplo de preprocesado — antes y después

In [6]:
print("EJEMPLO DE PREPROCESADO\n")
ejemplo = df['reviewText'].iloc[0]
print("- ORIGINAL:\n", ejemplo[:300])
print("\n- PREPROCESADO (sin stemming):\n", preprocess_text(ejemplo))
print("\n- PREPROCESADO (con stemming):\n", preprocess_text(ejemplo, stem=True))

EJEMPLO DE PREPROCESADO

- ORIGINAL:
 I put this bra in my Amazon cart months ago while looking for a cheaper version of my favorite brand (Moving Comfort: Fiona). After deciding to stick with what works, I essentially forgot about this item until months later when I accidentally purchased it thinking I had saved a Moving Comfort. When 

- PREPROCESADO (sin stemming):
 put bra amazon cart months ago looking cheaper version favorite brand moving comfort fiona deciding stick works essentially forgot item months later accidentally purchased thinking saved moving comfort bra showed angry made stupid mistake decided keep thought waste money boy wrong love bra fits great keeps ladies place something hard large busted lady find turned nice surprise number bra rotation

- PREPROCESADO (con stemming):
 put bra amazon cart month ago look cheaper version favorit brand move comfort fiona decid stick work essenti forgot item month later accident purchas think save move comfort bra show angri made s

## Aplicación al corpus completo

In [7]:
df['reviewText_clean'] = df['reviewText'].apply(preprocess_text)

print("Corpus preprocesado. Ejemplo:")
print(df[['reviewText', 'reviewText_clean']].head(3))

df.to_csv('corpus_preprocesado.csv', index=False)
print("\nGuardado como corpus_preprocesado.csv")

Corpus preprocesado. Ejemplo:
                                          reviewText  \
0  I put this bra in my Amazon cart months ago wh...   
1  I was a Brooks girl for several years (well a ...   
2  I have had these for about 4 months and I'm no...   

                                    reviewText_clean  
0  put bra amazon cart months ago looking cheaper...  
1  brooks girl several years well couple decades ...  
2  months im exactly sure sized gained little wei...  

Guardado como corpus_preprocesado.csv
